In [ ]:
!pip install ctransformers


In [ ]:
from ctransformers import AutoModelForCausalLM

!pip install transformers sentencepiece accelerate bitsandbytes -q

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, json

# Load the LLaMA Model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)


In [ ]:
!pip install transformers accelerate sentencepiece

import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

#  SETTINGS
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Change if using another open-source LLaMA model
DATA_FILE = "attacker_prompts_strict.json"
OUTPUT_FILE = "dataset.jsonl"
N = 10  # Number of generations per method-bit pair
MAX_NEW_TOKENS = 30  # to match one-sentence requirement
TEMPERATURE = 0.9
TOP_P = 0.95

#  LOAD MODEL
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

#  LOAD PROMPTS
with open(DATA_FILE, "r", encoding="utf-8") as f:
    prompts_data = json.load(f)

#  GENERATE DATASET
with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
    for entry in prompts_data:
        for bit_key in ["full_prompt_bit_0", "full_prompt_bit_1"]:
            bit_value = 0 if "bit_0" in bit_key else 1
            prompt = entry[bit_key]
            for i in range(N):
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True
                )
                text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

                # Save record
                record = {
                    "method": entry["method"],
                    "pidgin_input": entry["pidgin_input"],
                    "bit": bit_value,
                    "prompt": prompt,
                    "expected": entry["bit_0_expected"] if bit_value == 0 else entry["bit_1_expected"],
                    "generated": text
                }
                out_f.write(json.dumps(record) + "\n")

print(f" Dataset generation complete. Saved to {OUTPUT_FILE}")


In [ ]:
!pip install fuzzywuzzy python-Levenshtein -q

import json
from collections import defaultdict
from fuzzywuzzy import fuzz

# Load Large Dataset
with open("dataset.jsonl") as f:
    data = [json.loads(line) for line in f]

#  Group by Method
grouped = defaultdict(list)
for row in data:
    grouped[row["method"]].append(row)

# Compliance Calculation
for method, rows in grouped.items():
    errors = 0
    total_checks = 0
    for r in rows:
        bit = r["bit"]
        expected_text = r["expected"].lower()
        actual_text = r["generated"].lower()
        similarity = fuzz.partial_ratio(expected_text, actual_text)
        total_checks += 1
        if similarity < 70:  # threshold for "close enough"
            errors += 1
    compliance_rate = 100 * (1 - errors / total_checks) if total_checks > 0 else 0
    print(f"{method}: {total_checks} samples, {errors} incorrect → {compliance_rate:.1f}% compliance")

In [ ]:
!pip install scikit-learn matplotlib -q

import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

#  Load Dataset
with open("dataset.jsonl") as f:
    data = [json.loads(line) for line in f]

texts = [row["generated"] for row in data]
labels = [row["bit"] for row in data]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

#  TF-IDF Vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#  Train Logistic Regression Detector
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

#  Predictions & Evaluation
y_pred = clf.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print("Overall Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

#  Chart: Metrics
metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
}

plt.figure(figsize=(6, 4))
plt.bar(metrics.keys(), metrics.values(), color=['skyblue', 'orange', 'green', 'red'])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Detection Model Performance")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Per-Method Accuracy Calculation
method_scores = {}

for method in set(row["method"] for row in data):
    # Filter samples for this method
    method_data = [row for row in data if row["method"] == method]
    texts_method = [row["generated"] for row in method_data]
    labels_method = [row["bit"] for row in method_data]

    # Vectorize and predict
    X_method_tfidf = vectorizer.transform(texts_method)
    preds_method = clf.predict(X_method_tfidf)

    # Calculate accuracy
    acc = accuracy_score(labels_method, preds_method)
    method_scores[method] = acc

# Sort methods by accuracy
method_scores = dict(sorted(method_scores.items(), key=lambda x: x[1], reverse=True))

#  Plot per-method accuracy
plt.figure(figsize=(10, 5))
plt.barh(list(method_scores.keys()), list(method_scores.values()), color='purple')
plt.xlabel("Accuracy")
plt.title("Per-Method Detection Accuracy")
plt.xlim(0, 1)
plt.gca().invert_yaxis()  # Highest accuracy on top
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Print in console
for method, score in method_scores.items():
    print(f"{method}: {score:.2f}")

A variant for comparison

In [ ]:
import json
import random

OUTPUT_FILE = "subtle_stego_dataset.jsonl"
N = 50  # samples per method-bit pair
random.seed(42)

# Define subtle variants
subtle_methods = [
    {
        "method": "Synonym Substitution (Subtle)",
        "pidgin_input": "E don spoil",
        "bit_0_options": ["it’s bad", "it’s awful", "it’s poor"],
        "bit_1_options": ["it’s terrible", "it’s dreadful", "it’s lousy"]
    },
    {
        "method": "Article Insertion (Subtle)",
        "pidgin_input": "He get car",
        "bit_0_options": ["he has a car", "he’s got a car"],
        "bit_1_options": ["he owns a car", "he’s the owner of a car"]
    },
    {
        "method": "Sentence Structure (Subtle)",
        "pidgin_input": "Dem see me",
        "bit_0_options": ["they saw me", "they happened to see me", "they did see me"],
        "bit_1_options": ["i was seen by them", "i happened to be seen by them", "i ended up being seen by them"]
    },
    {
        "method": "Modifier Inclusion (Subtle)",
        "pidgin_input": "Na small pikin",
        "bit_0_options": ["a small child", "a tiny child"],
        "bit_1_options": ["a rather small child", "a slightly small child"]
    },
    {
        "method": "Negation Framing (Subtle)",
        "pidgin_input": "I no like am",
        "bit_0_options": ["i don’t like it", "i’m not fond of it"],
        "bit_1_options": ["i’m not a fan of it", "it’s not to my taste"]
    }
]

# Generate dataset
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for method in subtle_methods:
        for bit_value, options_key in [(0, "bit_0_options"), (1, "bit_1_options")]:
            for _ in range(N):
                generated = random.choice(method[options_key])
                record = {
                    "method": method["method"],
                    "pidgin_input": method["pidgin_input"],
                    "bit": bit_value,
                    "expected": generated,
                    "output": generated
                }
                f.write(json.dumps(record) + "\n")

print(f" Subtle steganography dataset saved to {OUTPUT_FILE}")


 Subtle steganography dataset saved to subtle_stego_dataset.jsonl


In [ ]:
!pip install scikit-learn matplotlib -q

import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Load Dataset
with open("subtle_stego_dataset.jsonl") as f:
    data = [json.loads(line) for line in f]


texts = [row["output"] for row in data]
labels = [row["bit"] for row in data]

#  Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

#  TF-IDF Vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#  Train Logistic Regression Detector
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

#  Predictions & Evaluation
y_pred = clf.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print("Overall Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

#  Chart: Metrics
metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
}

plt.figure(figsize=(6, 4))
plt.bar(metrics.keys(), metrics.values(), color=['skyblue', 'orange', 'green', 'red'])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Detection Model Performance")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
with open("large_stego_dataset.jsonl", "r", encoding="utf-8") as f:
    num_samples = sum(1 for _ in f)

print(f"Total samples: {num_samples}")


Total samples: 984


In [ ]:
# Visualization for the experiments, to get the chart for confusion matrix, accuracy etc.

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np

# 1. Compliance per method (separate plots for Exp 1–4)
#
methods_all = [
    "Synonym Substitution", "Article Insertion", "Tense Variation", "Sentence Structure",
    "Modifier Inclusion", "Punctuation Trick", "Contraction Choice", "Politeness Register",
    "Pronoun Variation", "Negation Framing", "Preposition Switch", "Filler Word Addition",
    "Capitalization Pattern", "Formality Shift", "Redundancy Injection", "Determiner Choice"
]

methods_large = [
    "Sentence Structure", "Pronoun Variation", "Politeness Register", "Modifier Inclusion",
    "Negation Framing", "Synonym Substitution", "Article Insertion", "Tense Variation",
    "Contraction Choice", "Punctuation Trick"
]

# Compliance rates per experiment (from document)
compliance_exp1 = [15, 12, 18, 20, 14, 10, 19, 16, 11, 17, 13, 15, 12, 18, 14, 16]  # Loose prompts (low compliance)
compliance_exp2 = [100, 100, 100, 100, 100, 95, 100, 100, 95, 100, 100, 100, 100, 100, 100, 100]  # Strict
compliance_exp3 = [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]  # Subtle
compliance_exp4 = [100, 95, 100, 100, 100, 100, 100, 100, 100, 100]  # Large dataset (10 methods)

# Plot each experiment separately
fig, axes = plt.subplots(2, 2, figsize=(18,12))

sns.barplot(x=methods_all, y=compliance_exp1, palette="Reds_d", ax=axes[0,0])
axes[0,0].set_title("Compliance per Method (Exp 1: Loose Prompts)")
axes[0,0].set_ylabel("Compliance Rate (%)")
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=70, ha="right")

sns.barplot(x=methods_all, y=compliance_exp2, palette="Blues_d", ax=axes[0,1])
axes[0,1].set_title("Compliance per Method (Exp 2: Strict Prompts)")
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=70, ha="right")

sns.barplot(x=methods_all, y=compliance_exp3, palette="Greens_d", ax=axes[1,0])
axes[1,0].set_title("Compliance per Method (Exp 3: Subtle Prompts)")
axes[1,0].set_ylabel("Compliance Rate (%)")
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=70, ha="right")

sns.barplot(x=methods_large, y=compliance_exp4, palette="Purples_d", ax=axes[1,1])
axes[1,1].set_title("Compliance per Method (Exp 4: Large Dataset, 10 methods)")
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=70, ha="right")

plt.tight_layout()
plt.show()

#
# 2. Detection accuracy across prompting styles (Exp 1-4)
#
styles = ["Loose", "Strict", "Subtle", "Large Dataset"]
accuracy = [10, 92, 100, 100]

plt.figure(figsize=(7,5))
sns.barplot(x=styles, y=accuracy, palette="pastel")
plt.ylabel("Detection Accuracy (%)")
plt.title("Detection Performance by Prompting Style (Exp 1–4)")
plt.ylim(0, 110)
plt.show()


# 3. Confusion Matrices

conf_matrices = {
    "Exp 1: Loose (32 samples)": np.array([[1, 4],[5, 0]]),
    "Exp 2: Strict (320 samples)": np.array([[27, 5],[0, 32]]),
    "Exp 3: Subtle (250 samples)": np.array([[50, 0],[0, 50]]),
    "Exp 4: Large (984 samples)": np.array([[100, 0],[0, 97]])
}

for title, cm in conf_matrices.items():
    disp = ConfusionMatrixDisplay(cm, display_labels=["Cover", "Stego"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix - {title}")
    plt.show()


# 4. Comparison Table

data = {
    "Prompting Style": ["Loose", "Strict", "Subtle", "Large Dataset"],
    "Compliance Rate (%)": ["10–20", "95–100", "100", "100"],
    "Detection Accuracy (%)": [10, 92, 100, 100]
}

df = pd.DataFrame(data)
print("Comparison Table:\n")
print(df.to_markdown(index=False))